In [13]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import time
import os
import logging
from datetime import datetime
from io import StringIO

In [14]:
#%pip install pyarrow
#%pip install lxml

In [ ]:
#Setting up logging and directories
os.makedirs("data/raw", exist_ok=True)
os.makedirs("logs", exist_ok=True)

log_filename = f"logs/pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()  # also print to console
    ]
)
log = logging.getLogger(__name__)
log.info(f"Pipeline started — log file: {log_filename}")

2026-03-30 09:49:18 [INFO] Pipeline started — log file: logs/pipeline_20260330_094918.log


In [ ]:
#defining ticker lists for all except stocks
etf_tickers = {
    "US_Equity_Broad": ["SPY","IVV","VOO","VTI","IWM","IWB","IWF","IWD","MDY","IJH","IJR","ITOT","SCHB","SCHA","SCHX","VXF","QQQ","ONEQ","QQQM","VONE","VTWO","VTWG","VTWV"],
    "US_Equity_Sector": ["XLF","XLK","XLE","XLV","XLI","XLY","XLP","XLU","XLB","XLRE","XLC","VFH","VGT","VDE","VHT","VIS","VCR","VDC","VPU","VAW","VOX","ARKK","ARKW","ARKG","ARKF","ARKQ","DRIV","BOTZ","ROBO","AIQ"],
    "International_Equity": ["EFA","IEFA","VEA","EEM","IEMG","VWO","ACWI","VT","EWJ","EWG","EWU","EWC","EWA","EWZ","EWY","EWT","EWH","EWS","EWL","EWP","EWQ","EWI","EWD","EWN","FXI","MCHI","KWEB","CNYA","ASHR","GXC"],
    "Fixed_Income_Broad": ["AGG","BND","BNDX","BNDW","FBND","BOND","GBF","IUSB","SCHZ","SPAB","USAG","NUBD","TOTL"],
    "Fixed_Income_Treasury": ["TLT","IEF","SHY","GOVT","VGSH","VGIT","VGLT","SCHO","SCHR","SCHQ","SPTS","SPTI","SPTL","VFISX","VFITX","VUSTX","TBT","TMF","TMV","TBF"],
    "Fixed_Income_Corporate": ["LQD","VCIT","VCSH","VCLT","IGIB","IGSB","IGLB","USIG","SPIB","SPSB","SPLB","QLTA","CBND","FCOR"],
    "Fixed_Income_HighYield": ["HYG","JNK","USHY","FALN","HYLB","HYDB","SHYG","SJNK","BSJO","BSJP","BSJQ","BSJR","BSJS"],
    "Commodity": ["GLD","IAU","GLDM","SGOL","BAR","OUNZ","SLV","SIVR","PSLV","USO","BNO","UCO","DBO","UNG","BOIL","KOLD","DJP","PDBC","COMT","BCI","COMB","CORN","WEAT","SOYB","COW","NIB"],
    "Real_Estate": ["VNQ","IYR","SCHH","RWR","USRT","REZ","REM","MORT","BBRE","PPTY","INDS","HOMZ","ROOF","NURE"],
    "Dividend": ["VYM","DVY","HDV","DGRO","SCHD","SDY","RDVY","FVD","DGRW","LVHD","SPHD","FDVV","DHS","PFF"],
    "Factor_Smart_Beta": ["MTUM","VLUE","USMV","QUAL","SIZE","LRGF","SMLF","INTF","ICVT","DYNF","BATT","ESGU","ESGV","SUSL"],
    "Leveraged_Inverse": ["TQQQ","SQQQ","UPRO","SPXU","SSO","SDS","QLD","QID","TNA","TZA","UDOW","SDOW","SPXL","SPXS"],
    "Volatility_Alternatives": ["VIXY","UVXY","SVXY","VXX","VIXM","TVIX","TAIL","BTAL","HDGE","DBMF","KMLM","CTA"],
}
crypto_tickers = {
    "Layer1": ["BTC-USD","ETH-USD","SOL-USD","ADA-USD","AVAX-USD","DOT-USD","ATOM-USD","NEAR-USD","ALGO-USD","FTM-USD","ONE-USD","EGLD-USD","HBAR-USD","XTZ-USD","EOS-USD"],
    "Layer2": ["MATIC-USD","ARB-USD","OP-USD","IMX-USD","LRC-USD","METIS-USD","BOBA-USD","SKL-USD"],
    "DeFi": ["UNI-USD","AAVE-USD","MKR-USD","CRV-USD","COMP-USD","SNX-USD","YFI-USD","SUSHI-USD","BAL-USD","1INCH-USD","DYDX-USD","GMX-USD","LDO-USD","RPL-USD","FXS-USD"],
    "Exchange": ["BNB-USD","CRO-USD","FTT-USD","OKB-USD","HT-USD","KCS-USD","GT-USD","MX-USD"],
    "Stablecoin_adjacent": ["XRP-USD","XLM-USD","XDC-USD","CELO-USD"],
    "Meme": ["DOGE-USD","SHIB-USD","PEPE-USD","FLOKI-USD","BONE-USD","BABYDOGE-USD","WIF-USD","BONK-USD"],
    "Storage_Privacy": ["FIL-USD","AR-USD","SC-USD","STORJ-USD","XMR-USD","ZEC-USD","DASH-USD","SCRT-USD"],
    "Gaming_NFT": ["AXS-USD","SAND-USD","MANA-USD","ENJ-USD","GALA-USD","ILV-USD","ALICE-USD","SLP-USD","GODS-USD","YGG-USD"],
    "Oracle_Data": ["LINK-USD","BAND-USD","TRB-USD","API3-USD","UMA-USD"],
    "Legacy_Altcoin": ["LTC-USD","BCH-USD","ETC-USD","ZIL-USD","VET-USD","ICX-USD","WAVES-USD","NEO-USD","ONT-USD","QTUM-USD"],
}
bond_tickers = {
    "Treasury_Short": ["SHY","SCHO","SCHR","VGSH","VGIT","SPTS","SPTI","BIL","SGOV","CLTL","TBLL","TFLO","USFR"],
    "Treasury_Long": ["TLT","IEF","GOVT","VGLT","SPTL","SCHQ","TBT","TMF","TMV","TBF","EDV","ZROZ","VUSTX"],
    "Corporate_IG": ["LQD","VCIT","VCSH","VCLT","IGIB","IGSB","IGLB","USIG","SPIB","SPSB","SPLB","QLTA","CBND","FCOR","IBND","GHYG","FLOT","FLRN","ICSH"],
    "Corporate_HY": ["HYG","JNK","USHY","FALN","HYLB","HYDB","SHYG","SJNK","BSJO","BSJP","BSJQ","BSJR","BSJS","BSJT","BSJU","ANGL","HYEM","EMHY"],
    "Municipal": ["MUB","VTEB","TFI","HYD","SMB","SHM","ITM","MLN","HYMB","MUNI","IBMK","IBML","IBMM","IBMN"],
    "International": ["BNDX","IAGG","BWX","IGOV","ISHG","PICB","IBND","EMB","PCY","VWOB","LEMB","EBND","EMAG"],
    "Inflation_Protected": ["TIP","SCHP","STIP","VTIP","PBTP","LTPZ","TIPX","FIPDX","TDTT","TDTF"],
    "Aggregate": ["AGG","BND","BNDW","FBND","BOND","GBF","IUSB","SCHZ","SPAB","NUBD","TOTL","DIAL","DFCF"],
}

In [ ]:
#building a ticker to category ticker to category map for all tickers.
ticker_to_category = {}
for category, tickers in etf_tickers.items():
    for t in tickers:
        ticker_to_category[t] = category
for category, tickers in crypto_tickers.items():
    for t in tickers:
        ticker_to_category[t] = category
for category, tickers in bond_tickers.items():
    for t in tickers:
        ticker_to_category[t] = category
#fetching sp500 tickers from wikipedia.
try:
    log.info("Fetching S&P 500 tickers from Wikipedia...")
    headers  = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        headers=headers,
        timeout=30
    )
    response.raise_for_status()
    sp500         = pd.read_html(StringIO(response.text))[0]
    sp500_tickers = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()
    for _, row in sp500.iterrows():
        ticker = row["Symbol"].replace(".", "-")
        ticker_to_category[ticker] = row["GICS Sector"]
    log.info(f"S&P 500 fetched — {len(sp500_tickers)} tickers")
except requests.exceptions.Timeout:
    log.error("Timeout fetching S&P 500 tickers — skipping stocks")
    sp500_tickers = []
except requests.exceptions.RequestException as e:
    log.error(f"Network error fetching S&P 500: {e} — skipping stocks")
    sp500_tickers = []
except Exception as e:
    log.error(f"Unexpected error fetching S&P 500: {e} — skipping stocks")
    sp500_tickers = []

log.info(f"Category map built — {len(ticker_to_category)} tickers")

2026-03-30 09:49:18 [INFO] Fetching S&P 500 tickers from Wikipedia...


2026-03-30 09:49:19 [INFO] S&P 500 fetched — 503 tickers
2026-03-30 09:49:19 [INFO] Category map built — 886 tickers


In [ ]:
#combine all tickers into a single dictionary and changing the formatting so that features are on the columns rather than rows.
def download_and_convert(tickers_dict, asset_class, start, end, chunk_size=50):
    all_dfs        = []
    failed_chunks  = []
    failed_tickers = []
    items = list(tickers_dict.items()) if isinstance(tickers_dict, dict) else [("default", tickers_dict)]

    for category, tickers in items:
        chunks = [tickers[i:i+chunk_size] for i in range(0, len(tickers), chunk_size)]
        for i, chunk in enumerate(chunks):
            log.info(f"[{asset_class}] {category} — chunk {i+1}/{len(chunks)} ({len(chunk)} tickers)")
            try:
                df = yf.download(chunk, start=start, end=end, auto_adjust=True, progress=False)
                if df.empty:
                    log.warning(f"[{asset_class}] {category} chunk {i+1} returned empty DataFrame")
                    failed_chunks.append((category, chunk))
                    continue

                long = df.stack(level=1, future_stack=True).reset_index()
                long.columns.name = None
                long = long.rename(columns={"level_0": "Date", "level_1": "Ticker"})
                price_cols = [c for c in ["Open","High","Low","Close","Volume"] if c in long.columns]

                if not price_cols:
                    log.warning(f"[{asset_class}] {category} chunk {i+1} — no price columns found, skipping")
                    failed_chunks.append((category, chunk))
                    continue

                long = long.dropna(subset=price_cols, how="all")
                long = long[["Date","Ticker"] + price_cols]
                long["AssetClass"] = asset_class
                long["Category"]   = long["Ticker"].map(ticker_to_category)

                # Flag any tickers that came back with all NaNs
                null_tickers = long.groupby("Ticker")["Close"].apply(lambda x: x.isna().all())
                for tkr, all_null in null_tickers.items():
                    if all_null:
                        log.warning(f"[{asset_class}] Ticker {tkr} has all-null Close prices")
                        failed_tickers.append(tkr)

                all_dfs.append(long)
                log.info(f"[{asset_class}] {category} chunk {i+1} — {len(long):,} rows loaded")

            except Exception as e:
                log.error(f"[{asset_class}] {category} chunk {i+1} failed: {e}", exc_info=True)
                failed_chunks.append((category, chunk))

            time.sleep(2)

    if failed_chunks:
        log.warning(f"[{asset_class}] {len(failed_chunks)} chunk(s) failed: "
                    + ", ".join(f"{cat}[{i}]" for i, (cat, _) in enumerate(failed_chunks)))
    if failed_tickers:
        log.warning(f"[{asset_class}] {len(failed_tickers)} ticker(s) with all-null data: {failed_tickers}")

    if not all_dfs:
        log.error(f"[{asset_class}] No data downloaded — returning empty DataFrame")
        return pd.DataFrame()

    result = pd.concat(all_dfs)
    log.info(f"[{asset_class}] Download complete — {len(result):,} total rows")
    return result


In [ ]:
#saving the data to parquet files.
def save_parquet(df, path, label):
    """Save DataFrame to parquet with error handling and size logging."""
    try:
        if df.empty:
            log.error(f"Skipping save for {label} — DataFrame is empty")
            return False
        df.to_parquet(path, index=False)
        size_gb = os.path.getsize(path) / 1e9
        log.info(f"Saved {label}: {len(df):,} rows — {size_gb:.3f} GB → {path}")
        return True
    except Exception as e:
        log.error(f"Failed to save {label} to {path}: {e}", exc_info=True)
        return False


In [ ]:
#logging for each asset class.
log.info("=" * 60)
log.info("DOWNLOADING ETFs")
log.info("=" * 60)
etf_long = download_and_convert(etf_tickers, "ETF", "1990-01-01", "2024-12-31")
save_parquet(etf_long, "data/etf_prices.parquet", "ETF prices")

log.info("=" * 60)
log.info("DOWNLOADING CRYPTO")
log.info("=" * 60)
crypto_long = download_and_convert(crypto_tickers, "Crypto", "2015-01-01", "2024-12-31")
save_parquet(crypto_long, "data/crypto_prices.parquet", "Crypto prices")

log.info("=" * 60)
log.info("DOWNLOADING BONDS")
log.info("=" * 60)
bond_long = download_and_convert(bond_tickers, "Bond", "1990-01-01", "2024-12-31")
save_parquet(bond_long, "data/bond_prices.parquet", "Bond prices")

log.info("=" * 60)
log.info("DOWNLOADING STOCKS (S&P 500)")
log.info("=" * 60)
if sp500_tickers:
    stock_chunks = {f"chunk_{i}": sp500_tickers[i:i+50] for i in range(0, len(sp500_tickers), 50)}
    stock_long   = download_and_convert(stock_chunks, "Stock", "1990-01-01", "2024-12-31", chunk_size=50)
    save_parquet(stock_long, "data/stock_prices.parquet", "Stock prices")
else:
    log.error("No S&P 500 tickers available — skipping stock download")
    stock_long = pd.DataFrame()

2026-03-30 09:49:19 [INFO] ============================================================
2026-03-30 09:49:19 [INFO] DOWNLOADING ETFs
2026-03-30 09:49:19 [INFO] ============================================================
2026-03-30 09:49:19 [INFO] [ETF] US_Equity_Broad — chunk 1/1 (23 tickers)
2026-03-30 09:49:20 [INFO] [ETF] US_Equity_Broad chunk 1 — 118,114 rows loaded
2026-03-30 09:49:22 [INFO] [ETF] US_Equity_Sector — chunk 1/1 (30 tickers)
2026-03-30 09:49:23 [INFO] [ETF] US_Equity_Sector chunk 1 — 135,058 rows loaded
2026-03-30 09:49:25 [INFO] [ETF] International_Equity — chunk 1/1 (30 tickers)
2026-03-30 09:49:27 [INFO] [ETF] International_Equity chunk 1 — 168,784 rows loaded
2026-03-30 09:49:29 [INFO] [ETF] Fixed_Income_Broad — chunk 1/1 (13 tickers)
2026-03-30 09:49:30 [INFO] [ETF] Fixed_Income_Broad chunk 1 — 41,010 rows loaded
2026-03-30 09:49:32 [INFO] [ETF] Fixed_Income_Treasury — chunk 1/1 (20 tickers)
2026-03-30 09:49:33 [INFO] [ETF] Fixed_Income_Treasury chunk 1 — 93,727

In [ ]:
#building the asset table, a master list of all tickers with their asset class and category.
log.info("=" * 60)
log.info("BUILDING ASSET TABLE")
log.info("=" * 60)
try:
    available = [df for df in [etf_long, crypto_long, bond_long, stock_long] if not df.empty]
    if not available:
        raise ValueError("All asset DataFrames are empty — cannot build asset table")

    all_prices = pd.concat(available)
    asset = (
        all_prices
        .groupby("Ticker")
        .agg(AssetClass=("AssetClass", "first"))
        .reset_index()
    )
    asset["Category"]      = asset["Ticker"].map(ticker_to_category)
    asset["Name"]          = None
    asset["Currency"]      = "USD"
    asset["InceptionDate"] = None
    save_parquet(asset, "data/asset_table.parquet", "Asset table")
    log.info(f"Asset table built — {len(asset)} tickers")
except Exception as e:
    log.error(f"Failed to build asset table: {e}", exc_info=True)

2026-03-30 09:51:37 [INFO] ============================================================
2026-03-30 09:51:37 [INFO] BUILDING ASSET TABLE
2026-03-30 09:51:37 [INFO] ============================================================
2026-03-30 09:51:37 [INFO] Saved Asset table: 884 rows — 0.000 GB → data/asset_table.parquet
2026-03-30 09:51:37 [INFO] Asset table built — 884 tickers


In [ ]:
#building the performance metrics table, meant to be a snapshot of key performance metrics.
log.info("=" * 60)
log.info("BUILDING PERFORMANCE METRICS")
log.info("=" * 60)

def calculate_metrics(df, ticker, period_label, period_days):
    try:
        df     = df.tail(period_days)
        closes = df["Close"].dropna().values
        if len(closes) < 2:
            log.debug(f"Skipping {ticker} {period_label} — insufficient data ({len(closes)} rows)")
            return None

        daily_returns     = np.diff(closes) / closes[:-1]
        total_return      = (closes[-1] - closes[0]) / closes[0]
        years             = period_days / 252
        annualized_return = (1 + total_return) ** (1/years) - 1
        volatility        = np.std(daily_returns) * np.sqrt(252)
        sharpe            = (annualized_return - 0.04) / volatility if volatility > 0 else None

        peak, max_drawdown = closes[0], 0
        for price in closes:
            if price > peak:
                peak = price
            drawdown = (peak - price) / peak
            if drawdown > max_drawdown:
                max_drawdown = drawdown

        return {
            "Ticker":           ticker,
            "Period":           period_label,
            "TotalReturn":      round(total_return, 4),
            "AnnualizedReturn": round(annualized_return, 4),
            "Volatility":       round(volatility, 4),
            "SharpeRatio":      round(sharpe, 4) if sharpe is not None else None,
            "MaxDrawdown":      round(max_drawdown, 4),
        }
    except Exception as e:
        log.error(f"Error calculating metrics for {ticker} {period_label}: {e}", exc_info=True)
        return None

try:
    all_prices["Date"] = pd.to_datetime(all_prices["Date"])
    all_prices = all_prices.sort_values(["Ticker", "Date"])
    all_prices["CapitalGain"] = (all_prices["Close"] - all_prices["Open"]) / all_prices["Open"]
    all_prices = all_prices.replace([np.inf, -np.inf], np.nan).dropna(subset=["CapitalGain"])

    periods      = {"1Y": 252, "3Y": 756, "5Y": 1260, "10Y": 2520}
    tickers      = all_prices["Ticker"].unique()
    met_rows     = []
    failed_count = 0
    CHUNK        = 100

    log.info(f"Calculating metrics for {len(tickers)} tickers across {len(periods)} periods...")

    for i in range(0, len(tickers), CHUNK):
        chunk_tickers = tickers[i:i+CHUNK]
        log.info(f"  Metrics {i+1}–{min(i+CHUNK, len(tickers))} of {len(tickers)}")
        for ticker in chunk_tickers:
            tdf = all_prices[all_prices["Ticker"] == ticker][["Date", "Close"]].dropna()
            for label, days in periods.items():
                result = calculate_metrics(tdf, ticker, label, days)
                if result:
                    met_rows.append(result)
                else:
                    failed_count += 1

    log.info(f"Metrics complete — {len(met_rows):,} records, {failed_count} skipped")
    performance = pd.DataFrame(met_rows)
    save_parquet(performance, "data/performance_metrics.parquet", "Performance metrics")

except Exception as e:
    log.error(f"Failed to build performance metrics: {e}", exc_info=True)

2026-03-30 09:51:37 [INFO] ============================================================
2026-03-30 09:51:37 [INFO] BUILDING PERFORMANCE METRICS
2026-03-30 09:51:37 [INFO] ============================================================
2026-03-30 09:51:38 [INFO] Calculating metrics for 883 tickers across 4 periods...
2026-03-30 09:51:38 [INFO]   Metrics 1–100 of 883
2026-03-30 09:51:39 [INFO]   Metrics 101–200 of 883
2026-03-30 09:51:40 [INFO]   Metrics 201–300 of 883
2026-03-30 09:51:41 [INFO]   Metrics 301–400 of 883
2026-03-30 09:51:41 [INFO]   Metrics 401–500 of 883
2026-03-30 09:51:42 [INFO]   Metrics 501–600 of 883
2026-03-30 09:51:43 [INFO]   Metrics 601–700 of 883
2026-03-30 09:51:44 [INFO]   Metrics 701–800 of 883
2026-03-30 09:51:45 [INFO]   Metrics 801–883 of 883
2026-03-30 09:51:46 [INFO] Metrics complete — 3,532 records, 0 skipped
2026-03-30 09:51:46 [INFO] Saved Performance metrics: 3,532 rows — 0.000 GB → data/performance_metrics.parquet


In [ ]:
#final summary of all files created, along with their sizes.
log.info("=" * 60)
log.info("FINAL FILE SUMMARY")
log.info("=" * 60)
files_map = {
    "ETF prices":          "data/etf_prices.parquet",
    "Crypto prices":       "data/crypto_prices.parquet",
    "Bond prices":         "data/bond_prices.parquet",
    "Stock prices":        "data/stock_prices.parquet",
    "Asset table":         "data/asset_table.parquet",
    "Performance metrics": "data/performance_metrics.parquet",
}
total = 0
for name, path in files_map.items():
    try:
        size   = os.path.getsize(path) / 1e9
        total += size
        log.info(f"  {name}: {size:.3f} GB")
    except FileNotFoundError:
        log.warning(f"  {name}: FILE NOT FOUND — likely failed during pipeline")

log.info(f"  Total: {total:.3f} GB")
log.info(f"Pipeline finished — log saved to {log_filename}")

2026-03-30 09:51:46 [INFO] ============================================================
2026-03-30 09:51:46 [INFO] FINAL FILE SUMMARY
2026-03-30 09:51:46 [INFO] ============================================================
2026-03-30 09:51:46 [INFO]   ETF prices: 0.032 GB
2026-03-30 09:51:46 [INFO]   Crypto prices: 0.007 GB
2026-03-30 09:51:46 [INFO]   Bond prices: 0.013 GB
2026-03-30 09:51:46 [INFO]   Stock prices: 0.120 GB
2026-03-30 09:51:46 [INFO]   Asset table: 0.000 GB
2026-03-30 09:51:46 [INFO]   Performance metrics: 0.000 GB
2026-03-30 09:51:46 [INFO]   Total: 0.172 GB
2026-03-30 09:51:46 [INFO] Pipeline finished — log saved to logs/pipeline_20260330_094918.log
